# Chapter 9 — Ontologies and Natural Languages
### Notebook 3 · Exercises

*Book reference: Section 9.3*

The book's exercises, executable. Assertions are the marking scheme.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch09_toolkit as ch9
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

### Exercise R1 — Verbalise five axioms and verify each

Verbalise five axioms in English and confirm each recovers exactly.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
axioms = ch9.SAMPLE_AXIOMS[:5]
rows = [ch9.round_trips(a) for a in axioms]
print(pd.DataFrame(rows)[['axiom', 'sentence', 'ok']].to_string(index=False))
assert all(r['ok'] for r in rows)

### Exercise R2 — Show why the language must be controlled

Give three paraphrases of one axiom that a human would accept, and show how many the parser accepts. Draw the conclusion.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
paraphrases = [
    'Every giraffe is a herbivore.',
    'All giraffes are herbivores.',
    'Giraffes are herbivores.',
    'A giraffe is always a herbivore.',
]
for text in paraphrases:
    print(f'{str(ch9.parse_cnl(text)):46s} <- {text}')
accepted = sum(ch9.parse_cnl(t) is not None for t in paraphrases)
print(f'\n{accepted}/{len(paraphrases)} accepted')
assert accepted < len(paraphrases)
print('\nEnglish offers many ways to say one thing; a parser with an inverse can\n'
      'afford exactly one. That is the trade a CONTROLLED language makes:\n'
      'expressive range for invertibility. Widen the grammar and you lose the\n'
      'free grader this chapter is built on.')

### Exercise R3 — Report translation status for a release

Produce a per-language table a project manager could act on: coverage, missing terms, and how many axioms are affected by each missing term.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
original = {k: dict(v) for k, v in ch9.LEXICON.items()}
try:
    for term in ['Leaf', 'isPartOf']:
        ch9.LEXICON[term].pop('de', None)
    status = ch9.lexicon_coverage('de')
    print('coverage:', status['coverage'], 'missing:', status['missing'])
    rows = []
    for term in status['missing']:
        affected = [a for a in ch9.SAMPLE_AXIOMS
                    if term in (a.subject, a.filler, a.property)]
        rows.append({'missing term': term, 'axioms affected': len(affected)})
    print(pd.DataFrame(rows).to_string(index=False))
    assert sum(r['axioms affected'] for r in rows) >= 2
    print('\nRanking missing terms by axioms affected turns "finish the\n'
          'translation" into a prioritised list -- the same move Chapter 5 made\n'
          'with competency-question coverage.')
finally:
    ch9.LEXICON.clear(); ch9.LEXICON.update(original)

## Where this leaves you

You have a generator with an exact inverse, a measurement of what that inverse cannot see, and a per-language coverage report. Notebook 4 gives the job to an agent and asks the question every LLM engineer eventually asks: **when should I stop resampling?**